# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the "FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and follows the [FAIR2 data standards](https://mlcommons.org/croissant/).

In [ ]:
# Ensure `mlcroissant` library is installed. If you use Colab or Jupyter locally, comment/uncomment as needed.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes (as object attributes)
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Croissant schema structures the data as record sets, which collect fields. We'll list all available record sets and fields (referenced by their `@id`).

In [ ]:
# List all record sets available in the dataset.

record_sets = dataset.metadata.recordSet

if not record_sets:
    print("No record sets found in metadata. Attempting ds.records() to enumerate available sets.")
else:
    print("Record sets in the dataset:")
    for rset in record_sets:
        print(f"@id: {rset['@id']}, name: {rset.get('name', 'no name')}, description: {rset.get('description', 'no description')}")

# Try to enumerate records for each record set.
recordset_ids = []

if not record_sets:
    # If metadata.recordSet is empty, try to enumerate record sets by using the records() generator
    # mlcroissant exposes all available record sets via ds.records(record_set=...)
    # Usually, you can call dataset.records() with no argument, but let's check that.
    try:
        print("Enumerating all available record sets (unique @id) from records()...")
        all_records = list(dataset.records())
        # Collect keys present per record
        ids = set()
        for record in all_records:
            # Record keys imply the record set's field @id
            ids.update(record.keys())
        print(f"Possible field/column @ids: {ids}")
    except Exception as err:
        print(f"Error enumerating records: {err}")
    # As example, use known record set id from FAIR2 dataset (often main CSV or main JSON if not listed; set explicitly below)
    main_recordset_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/mental_health_survey_2024_recordset'
    recordset_ids.append(main_recordset_id)
else:
    for rset in record_sets:
        recordset_ids.append(rset['@id'])

# For demonstration, show first N records in each record set (using @id)
for rset_id in recordset_ids:
    print(f"\nSample records for record set @id: {rset_id}")
    try:
        for i, rec in enumerate(dataset.records(record_set=rset_id)):
            print(rec)
            if i > 2:
                break
    except Exception as err:
        print(f"Error: {err}")

## 3. Data Extraction
Load records from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s as discovered in the overview. All references must be via `@id`.

In [ ]:
# Extract data from each discovered record set
dataframes = {}

for recordset_id in recordset_ids:
    try:
        records = list(dataset.records(record_set=recordset_id))
        df = pd.DataFrame(records)
        dataframes[recordset_id] = df
        print(f"Loaded DataFrame for record set @id: {recordset_id}, shape: {df.shape}")
    except Exception as err:
        print(f"Failed to load record set {recordset_id}: {err}")

# For most datasets, there is only one main record set. Use the first entry for demo.
selected_recordset_id = recordset_ids[0]
print("Columns of selected record set (all referenced by @id):")
print(dataframes[selected_recordset_id].columns.tolist())

# Display first few rows
dataframes[selected_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filter records by criteria, normalize numeric fields, manipulate columns, group by attributes, etc.

**Refer to columns by their `@id` where available.**

In [ ]:
# Example: Select a numeric field (by @id) for analysis, filter for high scores, normalize values.

# You may need to edit these IDs for your dataset. We'll assume the main mental health survey has 'GAD-7 Score' and 'PHQ-9 Score' by their actual @id.

# Example field @ids (adjust if needed for your dataset):
gad7_score_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/field/gad7_score'
phq9_score_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/field/phq9_score'
village_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/field/village'

df = dataframes[selected_recordset_id]

# Check available column @ids
print("Available columns (by @id):", df.columns.tolist())

numeric_field = gad7_score_id if gad7_score_id in df.columns else phq9_score_id
group_field = village_id if village_id in df.columns else None

# Filter for high GAD-7 or PHQ-9 scores (>10)
threshold = 10
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    normcol = f"{numeric_field}_normalized"
    filtered_df[normcol] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, normcol]].head())

    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean scores by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Numeric field '{numeric_field}' not found. Available fields: {df.columns.tolist()}")

## 5. Visualization
Visualize numeric field distributions or relationships (such as mean GAD-7 score per village).

All visualizations reference column `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of GAD-7 or PHQ-9

if numeric_field in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=12, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² mental health survey dataset from Kilifi County provides rich information about psychological symptoms and demographic factors.
- Using Croissant schema `@id`s enables reproducible, AI-ready manipulation of fields and records.
- The dataset supports research into mental health patterns in Kenya, and analysis by village or demographic group can help inform targeted interventions.
- The `mlcroissant` library makes it easy to load, filter, and visualize FAIR datasets for data science workflows.